# RFP Proposal Generator v2
### One-Shot Learning → Section Generation → IDML Template Injection → PDF Export

**Workflow (run each cell top-to-bottom):**
1. **Setup** — Install packages, configure paths, load firm info
2. **PDF Extraction** — Pull text from the example proposal and the target RFQ
3. **Claude Client** — Initialize with system prompt and cached message builder
4. **Section Generation** — One cell per proposal section; each streams from Claude and returns structured JSON
5. **IDML Injection** — XML helper utilities + story builders map generated JSON → template stories
6. **Build & Verify** — Write the modified IDML, spot-check stories
7. **Export PDF** — AppleScript triggers InDesign to export `example_rfq_to_rfp_proposal.pdf`

> **To use for a new RFQ:** update `RFQ_PDF` in Cell 2 and re-run from top.


In [17]:
%pip install -q anthropic pdfplumber



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import os, json, re, html, zipfile, subprocess
from pathlib import Path
from datetime import date
import anthropic
import pdfplumber
from IPython.display import display, Markdown

# ── API Key ────────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# ── Path resolver (handles iCloud Drive / macOS special paths) ─────────────────
def _find(filename):
    """Locate a file by name under the home directory via find."""
    r = subprocess.run(
        ['find', str(Path.home()), '-name', filename,
         '-not', '-path', '*/.Trash/*', '-not', '-path', '*/.git/*'],
        capture_output=True, text=True
    )
    hits = [l for l in r.stdout.strip().split('\n') if l.strip()]
    return Path(hits[0]) if hits else None

# ── File Paths — update RFQ_PDF for each new bid ──────────────────────────────
EXAMPLE_PDF   = _find("RFP 2025-22_Collab Architecture_civic_example_1.pdf")
RFQ_PDF       = _find("RFQ - AE Services for Cascade Campus Renovation Bid 2025-075-Final.pdf")
IDML_TEMPLATE = _find("2026 Master Template_Facing Pages.idml")

_HERE        = Path.cwd()
IDML_OUTPUT  = _HERE / "rfp_filled_template_v2.idml"
PDF_OUTPUT   = _HERE / "example_rfq_to_rfp_proposal.pdf"

# ── Status ─────────────────────────────────────────────────────────────────────
for label, val in [("Example PDF", EXAMPLE_PDF), ("RFQ PDF", RFQ_PDF), ("IDML template", IDML_TEMPLATE)]:
    ok = val and val.exists()
    print(f"{'✓' if ok else '✗'}  {label:15s}: {val}")

print(f"   Output PDF   : {PDF_OUTPUT}")
print(f"   API key      : {'✓ set' if ANTHROPIC_API_KEY else '✗ NOT SET — export ANTHROPIC_API_KEY=sk-ant-...'}")


✓  Example PDF    : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFP 2025-22_Collab Architecture_civic_example_1.pdf
✓  RFQ PDF        : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFQ - AE Services for Cascade Campus Renovation Bid 2025-075-Final.pdf
✓  IDML template  : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/Templates/InDesign Template_All Files/2026 Master Template_Facing Pages.idml
   Output PDF   : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/example_rfq_to_rfp_proposal.pdf
   API key      : ✓ set


In [19]:
# ── Firm & Team Information ────────────────────────────────────────────────────
# Update this block when submitting for a different firm.

FIRM_INFO = """
FIRM NAME:          Collab Architecture
ADDRESS:            9217 Eastman Park Drive, Unit 3, Windsor, CO 80550
PHONE:              970-292-7078
EMAIL:              jordan@collabarchitects.com
WEBSITE:            www.collabarchitects.com
YEAR FOUNDED:       2020
DISCIPLINES:        Architecture, Interior Design
RECOGNITION:        Named one of the fastest-growing private companies in Northern Colorado
                    and the Front Range (2023 & 2024)

PRIMARY CONTACT:
  Jordan W. Lockner, AIA, NCARB | jordan@collabarchitects.com | 970.215.9907

KEY PERSONNEL:
  - Jordan W. Lockner, AIA, NCARB
      Role:          Principal Architect / Primary Contact
      Education:     University of Colorado, B.ENVD (Architecture focus)
      Registrations: Licensed Architect, NCARB
      Awards:        2022 UofC ENVD Young Designer Award; 2023 N. Colorado 40 Under 40

  - Kala Bailor, AIA, LEED GA
      Role:          Project Manager / Primary Project Contact
      Education:     University of Colorado - Denver, Master of Architecture
      Registrations: Licensed Architect, LEED Green Associate

  - Michael Aller, AIA, LEED AP  ("Mick")
      Role:          QA/QC Manager
      Education:     University of Michigan, Master of Architecture
      Registrations: Licensed Architect, NCARB, LEED Accredited Professional
      Experience:    40+ years in municipal and higher education facility design
      Awards:        AIA Colorado Citation Award; F.W. Dodge Silver Hard Hat Award

SUB-CONSULTANTS:
  LARSEN STRUCTURAL DESIGN - Structural Engineering
      Blake Larsen, PE, LEED AP (19 yrs N. Colorado) | Fort Collins, CO

  INTEGRATED MEP - Mechanical, Electrical & Plumbing Engineering
      Thomas Segelhorst, PE, LEED AP | Lawrence Smith, PE | Fort Collins, CO

  JENSEN HUGHES - Fire Safety & Code Consulting (As-Needed)
      David Wolf, PE

  DFH CONSULTING - Cost Estimation (As-Needed)
      David Hoffman, PE

NOTABLE PROJECTS (COLLAB ARCHITECTURE):
  - City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO (289,000 sf)
  - Town of Superior, Downtown Civic Space - Superior, CO
  - Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
  - Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
  - Adams County, Honnen Facility Conditions Assessment - Brighton, CO
  - Department of Public Safety, Admin & Training Facility - Windsor, CO
  - Eaton Public Library, Renovation & Addition - Eaton, CO
  - Town of Silverthorne, Recreation Center Expansion - Silverthorne, CO
  - Town of Estes Park, Transit Facility - Estes Park, CO
  - Weld County, Grounds Building Design - Greeley, CO

NOTABLE PROJECTS (TEAM - PREVIOUS FIRMS):
  - Town of Timnath, Police Services Building, 29,000 sf (Kala Bailor) - Timnath, CO
  - Town of Timnath, Town Center, 15,250 sf (Kala Bailor) - Timnath, CO
  - Town of Windsor, Public Works Campus, 51,500 sf (Jordan Lockner) - Windsor, CO
  - Larimer County Police and Courts Addition (Blake Larsen) - Loveland, CO
  - City of Loveland, Fire Station 3 (Thomas Segelhorst) - Loveland, CO
  - City of Loveland, Fire Station 4 (Thomas Segelhorst) - Loveland, CO

BILLING RATES:
  - Principal Architect / Engineer:  $225/hr
  - Project Architect / Engineer:    $205/hr
  - Project Manager / Engineer:      $185/hr
  - QA/QC Review:                    $185/hr
  - CAD Technician:                  $115/hr
  - Interior Designer:                $95/hr
  - Administrative:                   $75/hr

REIMBURSABLES:
  - Outside Materials / Services / Supplies:  Cost + 15%
  - Mileage:                                  $0.70 / mile

REFERENCES:
  - Robert Wynkoop, Police Sergeant, Town of Timnath
      970.224.3211 | rwynkoop@timnathgov.com
  - Brian Rowe, Deputy Director of Public Works, Town of Windsor
      970.674.5400 | browe@windsorgov.com
  - Elly Watson, Business Services Manager, City of Aurora
      303.739.7109 | elwatson@auroragov.org
  - Jordan Hayes, Parks & Recreation Analyst II, Town of Superior
      303.499.3675 | jordanh@superiorcolorado.gov
  - Kyle Burg, Project Manager, Facilities & Fleet Mgmt., Adams County
      720.523.6062 | KBurg@adcogov.org
"""
print("Firm information loaded.")


Firm information loaded.


## Step 1 — Extract PDF Text

Pull text from the one-shot example proposal and the target RFQ.  
Run these two cells before any generation cell.


In [20]:
# ── Extract One-Shot Example Proposal ─────────────────────────────────────────
def extract_pdf(path):
    """Extract all text from a PDF, page-labelled."""
    pages = []
    with pdfplumber.open(str(path)) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text and text.strip():
                pages.append(f"[Page {i+1}]\n{text.strip()}")
    return "\n\n".join(pages)

example_text = extract_pdf(EXAMPLE_PDF)
print(f"Example proposal: {len(example_text):,} chars across {example_text.count('[Page')} pages")
print("\nFirst 500 chars:")
print(example_text[:500], "...")


Example proposal: 41,725 chars across 19 pages

First 500 chars:
[Page 1]
RESPONSE TO RFP
RFP #2025-22
CITY OF LOVELAND
DESIGN SERVICES FOR POLICE AND
COURTS BUILDING RENOVATION
MARCH 20, 2025
970-292-7078 | WWW.COLLABARCHITECTS.COM | 9217 EASTMAN PARK DR. WINDSOR, CO 80550

[Page 2]
CITY OF LOVELAND
DESIGN SERVICES FOR POLICE AND COURTS BUILDING RENOVATION
RFP #2025-22
TABLE OF
CONTENTS
A./ COVER LETTER...........................................................................................3
B./ RELEVANT PROJECT EXPERIENCE.................................. ...


In [21]:
# ── Extract Target RFQ PDF ─────────────────────────────────────────────────────
# Re-run this cell alone when you swap in a new RFQ PDF.

rfq_text = extract_pdf(RFQ_PDF)
print(f"Target RFQ: {len(rfq_text):,} chars across {rfq_text.count('[Page')} pages")
print("\nFirst 500 chars:")
print(rfq_text[:500], "...")


Target RFQ: 36,978 chars across 17 pages

First 500 chars:
[Page 1]
REQUEST FOR QUALIFICATIONS
Architectural and Engineering Services
For
Cascade Campus Facility Renovation
Bid No. 2025-075
Owner:
City of Loveland
Public Works Department/Facilities Division
105 West 5th Street
Loveland, CO 80537
Issue Date:
December 5, 2025

[Page 2]
REQUEST FOR QUALIFICATIONS
The City of Loveland, Colorado (“City”) is seeking Statements of Qualifications (SOQ) from
qualified architectural and engineering firms (“Consultant”) to provide architectural and
engineering ser ...


## Step 2 — Initialize Claude Client

Sets up the streaming generator with a cached multi-turn message structure:
- **Turn 1** (cached): example proposal → Claude learns the style
- **Turn 2** (cached): RFQ + firm info → context for this bid
- **Turn 3**: the specific section task


In [22]:
# ── Claude Client + Cached Message Builder ────────────────────────────────────
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """\
You are a senior proposal writer for a professional architecture firm.
Generate compelling, client-focused proposal content from the RFQ and firm information.

RULES (non-negotiable):
- Output ONLY valid JSON — no markdown fences, no commentary outside the JSON object.
- Write in first-person plural: "our team", "we will", "we bring".
- Reference specific RFQ details: project name, scope, evaluation criteria, constraints.
- Name real team members and real past projects from the firm information provided.
- Match the professional tone of the example proposal you were shown.
- Do not invent facts; if a detail is unavailable, note what should be inserted.
"""


def _messages(task_prompt):
    """
    Three-turn cached message structure.
    After the first API call, subsequent calls read from cache (very low cost).
    """
    return [
        # Turn 1: one-shot style example (cached)
        {
            "role": "user",
            "content": [{
                "type": "text",
                "text": (
                    "Study this completed proposal carefully — match its professional "
                    "voice, structure, and level of detail precisely.\n\n"
                    "## COMPLETED EXAMPLE PROPOSAL\n\n" + example_text
                ),
                "cache_control": {"type": "ephemeral"},
            }],
        },
        {
            "role": "assistant",
            "content": (
                "I have carefully reviewed the example proposal and understand "
                "its tone, structure, and style. Ready to write."
            ),
        },
        # Turn 2: target RFQ + firm info (cached) + task
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"## TARGET RFQ\n\n{rfq_text}\n\n## FIRM INFORMATION\n\n{FIRM_INFO}",
                    "cache_control": {"type": "ephemeral"},
                },
                {"type": "text", "text": task_prompt},
            ],
        },
    ]


def generate(label, prompt, max_tokens=2000):
    """Stream one proposal section; return the full response string."""
    print(f"\n{'=' * 60}\n  {label}\n{'=' * 60}\n")
    result = ""
    with client.messages.stream(
        model="claude-opus-4-8",
        max_tokens=max_tokens,
        system=SYSTEM_PROMPT,
        messages=_messages(prompt),
    ) as stream:
        for chunk in stream.text_stream:
            print(chunk, end="", flush=True)
            result += chunk
        usage = stream.get_final_message().usage
    print(
        f"\n\n[tokens — in: {usage.input_tokens:,}  out: {usage.output_tokens:,}  "
        f"cache_read: {usage.cache_read_input_tokens:,}  "
        f"cache_write: {usage.cache_creation_input_tokens:,}]"
    )
    return result


def parse_json(raw):
    """Strip optional markdown fences and parse JSON."""
    clean = re.sub(r'^```[a-z]*\n?|\n?```$', '', raw.strip(), flags=re.MULTILINE)
    return json.loads(clean.strip())


print("Claude client ready. Model: claude-opus-4-8")
print("Run each section cell below independently — cached after the first call.")


Claude client ready. Model: claude-opus-4-8
Run each section cell below independently — cached after the first call.


## Step 3 — Generate Proposal Sections

Each cell below calls Claude once and returns structured JSON.  
The RFQ and example proposal are cached after the first call — subsequent sections cost only output tokens.

Run cells in order, or re-run any single section to regenerate it.


In [23]:
# ── Section 1: Cover Letter ───────────────────────────────────────────────────
# Returns: rfp_title, rfp_number, client info, re_line, salutation, body (4 paragraphs)

_cover_raw = generate(
    "COVER LETTER",
    f"""
Extract RFQ metadata and write a 4-paragraph cover letter body.
Return a JSON object with EXACTLY these keys (no extras):

  rfp_title       — full project / RFQ title (string)
  rfp_number      — bid or RFQ number, e.g. "Bid 2025-075" (string)
  client_name     — issuing organization name (string)
  client_dept     — department name or "" (string)
  client_address1 — street address (string)
  client_address2 — city, state, zip (string)
  re_line         — full "Re:" line text (string)
  salutation      — e.g. "Dear Selection Committee," (string)
  body            — exactly 4 flowing prose paragraphs separated by \\n\\n:
                    Para 1: understanding of the project scope and the client's core need
                    Para 2: why Collab Architecture is uniquely qualified (name 1-2 past projects)
                    Para 3: introduce Jordan Lockner (Principal), Kala Bailor (PM), Michael Aller (QA/QC)
                    Para 4: closing commitment and call to action (2 sentences)

Today is {date.today().strftime("%B %d, %Y")}.
Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=2000,
)

cover = parse_json(_cover_raw)
print("\n--- Parsed cover letter fields ---")
for k, v in cover.items():
    if k != "body":
        print(f"  {k:18s}: {str(v)[:80]}")
print(f"  {'body':18s}: {str(cover.get('body',''))[:100]}...")



  COVER LETTER

{
  "rfp_title": "Cascade Campus Facility Renovation",
  "rfp_number": "Bid No. 2025-075",
  "client_name": "City of Loveland",
  "client_dept": "Public Works Department / Facilities Division",
  "client_address1": "105 West 5th Street",
  "client_address2": "Loveland, CO 80537",
  "re_line": "Re: Bid No. 2025-075 — Architectural and Engineering Services for Cascade Campus Facility Renovation",
  "salutation": "To The City of Loveland Selection Committee & Stakeholders,",
  "body": "After reviewing the City of Loveland's Request for Qualifications, attending the mandatory pre-submittal meeting at 1515 Cascade Avenue, and evaluating the project's objectives, we recognize the significance of this renovation in supporting the long-term growth of the Loveland Utilities Department. Our team understands that this phase will modernize approximately 62,030 square feet across the first and second floors to accommodate the Department's 15-year staffing projections, encompassing 

In [24]:
# ── Section 2: Firm Profile ───────────────────────────────────────────────────
# Returns: headline, who_we_are (2-3 paragraphs), differentiator

_firm_raw = generate(
    "FIRM PROFILE",
    """
Write firm profile content tailored to this specific RFQ scope.
Return a JSON object with EXACTLY these keys:

  headline       — short display tagline (10 words max, no period) for the page header (string)
  who_we_are     — 2-3 flowing prose paragraphs (joined by \\n\\n) for the "WHO WE ARE"
                   section: introduce Collab Architecture, its founding year, philosophy,
                   growth recognition, and municipal/civic project expertise (string)
  differentiator — 1 focused prose paragraph on why Collab is uniquely right for
                   THIS specific project and client (string)

Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=2000,
)

firm = parse_json(_firm_raw)
print("\n--- Parsed firm profile ---")
print(f"  Headline:       {firm.get('headline','')}")
print(f"  Who we are:     {str(firm.get('who_we_are',''))[:120]}...")
print(f"  Differentiator: {str(firm.get('differentiator',''))[:100]}...")



  FIRM PROFILE

{
  "headline": "Collaborative Design for Loveland's Growing Utilities Workforce",
  "who_we_are": "Collab Architecture is a Windsor-based architecture and interior design firm dedicated to bringing the power of collaborative design to create a stronger, better, and more sustainable community. Founded in 2020, our firm has rapidly established itself as a trusted partner for municipal clients throughout Northern Colorado and the Front Range, and we have been named one of the fastest-growing private companies in the region in both 2023 and 2024. Our holistic approach to every project begins with two fundamental questions: What design problem are we looking to solve, and how can we make the project the most successful for our client? It all starts with listening — \"Stop. Collaborate and Listen\" is our unofficial motto, and we push to live by those words on every engagement.\n\nWe provide full-service commercial architecture and interior design, with in-depth experience 

In [25]:
# ── Section 3: Team Bios ──────────────────────────────────────────────────────
# Returns bios for Jordan Lockner, Kala Bailor, and Michael Aller.

_team_raw = generate(
    "TEAM BIOS",
    """
Write bio content for three key team members.
Return a JSON object with keys "lockner", "bailor", and "aller".
Each value is an object with EXACTLY these keys:

  name          — full name with credentials, e.g. "Jordan W. Lockner, AIA, NCARB" (string)
  title         — job title on THIS project (string)
  role_label    — role label line, e.g. "Project Role: Principal Architect / Primary Contact" (string)
  bio           — 3-4 sentence prose paragraph on their expertise and fit for this RFQ (string)
  education     — degree and institution, e.g. "University of Colorado, B.ENVD" (string)
  registrations — comma-separated licenses/credentials (string)
  experience    — JSON array of 6-8 strings, each "Client, Project Name - City, ST"
                  (pull from firm's documented project history; note "(previous firm)" where applicable)

Use ONLY facts from the firm information. Tailor each bio to this RFQ's scope.
Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=2500,
)

team = parse_json(_team_raw)
print("\n--- Parsed team bios ---")
for key in ["lockner", "bailor", "aller"]:
    m = team.get(key, {})
    print(f"  {key:10s}: {m.get('name','')}  |  {m.get('title','')}")
    print(f"             {len(m.get('experience',[]))} experience entries")



  TEAM BIOS

{
  "lockner": {
    "name": "Jordan W. Lockner, AIA, NCARB",
    "title": "Principal Architect",
    "role_label": "Project Role: Principal Architect / Primary Contact",
    "bio": "Jordan believes that the root of all good architecture must stem from and support the community it serves, an approach well-suited to the City of Loveland's vision for the Cascade Campus renovation in support of the growing Loveland Utilities Department. His ability to listen, understand, and collaborate with clients while coordinating and managing the design process has produced successful outcomes for municipal facilities throughout Northern Colorado, including the 51,500 SF Town of Windsor Public Works Campus and the 289,000 SF City of Aurora Municipal Center Space Planning effort. As Principal Architect and your primary firm contact, Jordan will provide strategic oversight to ensure the renovation aligns with the Department's 15-year staffing projections and the City's space standards. Hi

In [26]:
# ── Section 4: Relevant Project Experience ────────────────────────────────────
# Returns 2 project case studies most relevant to this RFQ.

_exp_raw = generate(
    "RELEVANT PROJECT EXPERIENCE",
    """
Select and write 2 project case studies most directly relevant to this RFQ.
Return a JSON object with keys "project1" and "project2". Each value has EXACTLY:

  title         — project name in UPPERCASE (string)
  location      — "City, ST" (string)
  team_lead     — Collab team member who led it (string)
  project_type  — e.g. "Interior Renovation, Space Planning" (string)
  services      — e.g. "Architecture, Interior Design, Construction Administration" (string)
  description   — 2-3 flowing prose paragraphs (joined by \\n\\n):
                  what was designed/built, why it is relevant to THIS RFQ scope,
                  one concrete outcome or measurable result

Select ONLY from the firm's documented past projects in the firm information.
Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=1500,
)

experience = parse_json(_exp_raw)
print("\n--- Parsed project experience ---")
for key in ["project1", "project2"]:
    p = experience.get(key, {})
    print(f"  {key}: {p.get('title','')[:65]}")
    print(f"         {p.get('location','')}  |  Lead: {p.get('team_lead','')}")



  RELEVANT PROJECT EXPERIENCE

{
  "project1": {
    "title": "CITY OF AURORA, MUNICIPAL CENTER SPACE PLANNING & REMODEL",
    "location": "Aurora, CO",
    "team_lead": "Jordan Lockner, AIA, NCARB",
    "project_type": "Interior Renovation, Space Planning, Department Reconfiguration",
    "services": "Architecture, Interior Design, Programming, Construction Administration",
    "description": "Our team began its partnership with the City of Aurora through a comprehensive space planning effort and audit of the 289,000-square-foot Aurora Municipal Center (AMC). This project required evaluating space utilization across five stories and 20 municipal departments to address the facility's full capacity and prepare for future growth. By assessing operational needs, hybrid work opportunities, and potential department consolidations, we created a road-map to enhance efficiency, streamline workflows, and support scalability within the AMC—precisely the kind of programming and multi-floor space

In [27]:
# ── Section 5: Project Understanding & Approach ───────────────────────────────
# Returns approach narrative, key challenges, schedule framing, and references.

_approach_raw = generate(
    "PROJECT UNDERSTANDING & APPROACH",
    """
Write project understanding and approach content.
Return a JSON object with EXACTLY these keys:

  intro           — 2 flowing prose paragraphs (joined by \\n\\n) showing deep understanding
                    of this RFQ: reference specific scope elements, constraints, evaluation
                    criteria, and stated client goals from the RFQ document (string)
  key_challenges  — 2-3 sentences identifying the single biggest project challenge
                    and our specific mitigation strategy (string)
  schedule_intro  — 1-2 sentences framing our proposed schedule relative to the RFQ's timeline (string)
  references      — JSON array of exactly 3 strings, each formatted as:
                    "Name, Title, Organization\\nPhone  |  Email"
                    Use actual references from the firm information.

Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=1800,
)

approach = parse_json(_approach_raw)
print("\n--- Parsed approach ---")
print(f"  Intro:           {str(approach.get('intro',''))[:100]}...")
print(f"  Key challenges:  {str(approach.get('key_challenges',''))[:100]}...")
print(f"  References:      {len(approach.get('references', []))} entries")
for ref in approach.get('references', []):
    print(f"    - {ref.split(chr(10))[0][:70]}")



  PROJECT UNDERSTANDING & APPROACH

{
  "intro": "After reviewing the City of Loveland's RFQ for the Cascade Campus Facility Renovation, attending the mandatory pre-submittal meeting, and evaluating the project's objectives, we recognize the strategic importance of transforming the newly acquired 1515 Cascade Avenue facility into a modern, functional home for the Loveland Utilities Department. This renovation of approximately 62,030 SF across the first and second floors represents a critical investment in supporting the Department's projected 15-year staffing growth, building upon the programming efforts the City initiated in 2023. Our team is prepared to deliver a thoughtful renovation that addresses the full scope outlined in the RFQ, including interior finish updates, space planning and furniture layouts per City standards, collaborative teaming and conference spaces with audio/visual improvements, break areas and outdoor amenities, comprehensive MEP and network connectivity upgrad

In [28]:
# ── Section 6: Fee Schedule ────────────────────────────────────────────────────
# Returns billing rates, reimbursables, and fee narrative.

_fee_raw = generate(
    "FEE SCHEDULE",
    """
Write fee schedule content.
Return a JSON object with EXACTLY these keys:

  narrative      — 2-3 sentences describing our fee approach: unit-price NTE basis,
                   phase-level budget caps, monthly earned-value reporting (string)
  rates          — JSON array of objects {"role": "...", "rate": "$X / hour"}
                   Use the ACTUAL billing rates from firm information (array)
  reimbursables  — JSON array of objects {"item": "...", "basis": "..."}
                   Use the ACTUAL reimbursable policy from firm information (array)

Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=1000,
)

fee = parse_json(_fee_raw)
print("\n--- Parsed fee schedule ---")
print(f"  Narrative:  {str(fee.get('narrative',''))[:100]}...")
print(f"  Rates ({len(fee.get('rates',[]))} roles):")
for r in fee.get('rates', []):
    print(f"    {r.get('role',''):35s}  {r.get('rate','')}")
print(f"  Reimbursables ({len(fee.get('reimbursables',[]))} items):")
for r in fee.get('reimbursables', []):
    print(f"    {r.get('item',''):35s}  {r.get('basis','')}")



  FEE SCHEDULE

{
  "narrative": "Our fee is structured on a unit-price basis with a not-to-exceed total aligned to the City of Loveland's purchasing requirements and the Cascade Campus Facility Renovation scope, from conceptual programming through project closeout. Each phase—conceptual design and programming, schematic design, design development, construction documents, bidding and procurement, and construction administration—carries its own budget cap, allowing the City to track financial commitments against deliverables at every milestone. We will provide monthly earned-value reporting that ties invoiced amounts to percentage of work completed, ensuring full transparency and proactive cost control throughout the design-bid-build process.",
  "rates": [
    {"role": "Principal Architect / Engineer", "rate": "$225 / hour"},
    {"role": "Project Architect / Engineer", "rate": "$205 / hour"},
    {"role": "Project Manager / Engineer", "rate": "$185 / hour"},
    {"role": "QA/QC Revie

## Step 4 — IDML Template Injection

The three cells below do not call Claude — they build XML and write files.

1. **Helper utilities** — XML escaping, Markdown stripping, story envelope
2. **Story builders** — One function per template section type
3. **Build IDML** — Applies all story updates and writes the modified `.idml` file


In [29]:
# ── IDML XML Helper Utilities ─────────────────────────────────────────────────

def xe(text):
    """XML-escape a string for IDML Content elements."""
    return html.escape(str(text), quote=False)

def strip_md(text):
    """Remove Markdown syntax, returning clean plain text."""
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*]+)\*', r'\1', text)
    text = re.sub(r'^\s*[-*]\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'^---+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\|[^\n]*\|[^\n]*\n', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def wrap_story(self_id, inner_xml):
    """Wrap inner paragraph XML in the standard IDML <Story> envelope."""
    return (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>\n'
        '<idPkg:Story xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging"'
        ' DOMVersion="21.2">\n'
        f'\t<Story Self="{self_id}" UserText="true" IsEndnoteStory="false"'
        ' AppliedTOCStyle="n" TrackChanges="false" StoryTitle="$ID/" AppliedNamedGrid="n">\n'
        '\t\t<StoryPreference OpticalMarginAlignment="false" OpticalMarginSize="12"'
        ' FrameType="TextFrameType" StoryOrientation="Horizontal"'
        ' StoryDirection="LeftToRightDirection" />\n'
        '\t\t<InCopyExportOption IncludeGraphicProxies="true"'
        ' IncludeAllResources="false" />\n'
        f'{inner_xml}\n'
        '\t</Story>\n'
        '</idPkg:Story>'
    )

def pb(text, style="Body", char_style="$ID/[No character style]", br=True):
    """Build one ParagraphStyleRange / CharacterStyleRange block."""
    br_tag = '\n\t\t\t\t<Br />' if br else ''
    return (
        f'\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/{xe(style)}">\n'
        f'\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/{xe(char_style)}">\n'
        f'\t\t\t\t<Content>{xe(text)}</Content>{br_tag}\n'
        f'\t\t\t</CharacterStyleRange>\n'
        f'\t\t</ParagraphStyleRange>'
    )

def pbs(text, style="Body"):
    """Convert multi-paragraph plain text into consecutive paragraph blocks."""
    chunks = [c.strip() for c in text.split('\n\n') if c.strip()]
    return '\n'.join(pb(c, style) for c in chunks)

def rc(xml, old, new):
    """Replace the first <Content>old</Content> in a story XML string."""
    return xml.replace(
        f'<Content>{html.escape(old, quote=False)}</Content>',
        f'<Content>{xe(new)}</Content>',
        1,
    )

print("IDML helper utilities loaded (xe, strip_md, wrap_story, pb, pbs, rc).")


IDML helper utilities loaded (xe, strip_md, wrap_story, pb, pbs, rc).


In [30]:
# ── Story XML Construction Functions ─────────────────────────────────────────
# Each function takes parsed JSON data and returns a complete IDML Story XML string.

def story_cover_letter(c):
    """Story u221 — full cover letter body with recipient block and signature."""
    body_chunks = [ch.strip() for ch in c['body'].split('\n\n') if ch.strip()]
    content = [
        f'\t\t\t\t<Content>{xe(c["salutation"])}</Content>\n\t\t\t\t<Br />',
        '\t\t\t\t<Br />',
    ]
    for ch in body_chunks:
        content += [f'\t\t\t\t<Content>{xe(ch)}</Content>', '\t\t\t\t<Br />', '\t\t\t\t<Br />']
    content += [
        '\t\t\t\t<Content>Sincerely,</Content>', '\t\t\t\t<Br />',
        '\t\t\t\t<Br />', '\t\t\t\t<Br />',
        '\t\t\t\t<Content>Jordan W. Lockner, AIA, NCARB</Content>', '\t\t\t\t<Br />',
        '\t\t\t\t<Content>Founding Principal  |  Collab Architecture</Content>', '\t\t\t\t<Br />',
        '\t\t\t\t<Content>jordan@collabarchitects.com  |  970.215.9907</Content>',
    ]
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body - no spacing">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(c["client_name"])}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>{xe(c.get("client_dept") or c["client_name"])}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>{xe(c["client_address1"])}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>{xe(c["client_address2"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        '\t\t\t\t<Br />\n\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Bold, Teal">\n'
        f'\t\t\t\t<Content>{xe(c["re_line"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        + '\n'.join(content) + '\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story('u221', inner)


def story_team_name(sid, m):
    """Team name + title stories: u67f (Jordan), u703 (Kala), u918 (Mick)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Name &amp; Credentials">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["name"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Job Title">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["title"])}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_team_bio(sid, m):
    """Team role + bio stories: u665 (Jordan), u6ea (Kala), u8b3 (Mick)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Headers">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["role_label"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(strip_md(m["bio"]))}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_team_edu(sid, m):
    """Team education + registrations stories: u64c (Jordan), u6d0 (Kala), u84e (Mick)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Headers">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Resumes:Resume Headers">\n'
        '\t\t\t\t<Content>Education</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["education"])}</Content>\n\t\t\t\t<Br />\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Resumes:Resume Headers">\n'
        '\t\t\t\t<Content>Registrations &amp; Affiliations</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["registrations"])}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_team_exp(sid, m):
    """Team select experience stories: u69e (Jordan), u71f (Kala), u986 (Mick)."""
    exp_lines = '\n'.join(
        f'\t\t\t\t<Content>{xe(p)}</Content>\n\t\t\t\t<Br />'
        for p in m.get('experience', [])
    )
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Headers">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Resumes:Resume Headers">\n'
        '\t\t\t\t<Content>Select Experience</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'{exp_lines}\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_proj_title(sid, p):
    """Project title + location stories: u2ef (project 1), u3eb (project 2)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Experience:Project Name">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(p["title"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Experience:City, State">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(p["location"])}  |  Lead: {xe(p.get("team_lead",""))}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_proj_desc(sid, p):
    """Project description stories: u309 (project 1), u404 (project 2)."""
    chunks = [c.strip() for c in p['description'].split('\n\n') if c.strip()]
    blocks = '\n'.join(
        f'\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body">\n'
        f'\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(c)}</Content>\n'
        f'\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
        for c in chunks
    )
    return wrap_story(sid, blocks)


def story_proj_meta(sid, p):
    """Project type + services stories: u322 (project 1), u41d (project 2)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body - no spacing">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>Type of Project: {xe(p.get("project_type",""))}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>Services Provided: {xe(p.get("services",""))}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_references(refs):
    """References story u1552."""
    lines = []
    for ref in refs:
        for line in ref.split('\n'):
            if line.strip():
                lines.append(f'\t\t\t\t<Content>{xe(line.strip())}</Content>\n\t\t\t\t<Br />')
        lines.append('\t\t\t\t<Br />')
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        + '\n'.join(lines) + '\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story('u1552', inner)


def story_billing_rates(fee):
    """Standard hourly rates story u1d57."""
    rate_lines = []
    for r in fee.get('rates', []):
        rate_lines += [
            f'\t\t\t\t<Content>{xe(r.get("role",""))}</Content>\n\t\t\t\t<Br />',
            f'\t\t\t\t<Content>{xe(r.get("rate",""))}</Content>\n\t\t\t\t<Br />',
        ]
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body - no spacing">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Bold, Teal">\n'
        '\t\t\t\t<Content>STANDARD HOURLY RATES</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        + '\n'.join(rate_lines) + '\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story('u1d57', inner)


print("All story builder functions loaded.")
print("Builders: story_cover_letter, story_team_name/bio/edu/exp, story_proj_title/desc/meta,")
print("          story_references, story_billing_rates")


All story builder functions loaded.
Builders: story_cover_letter, story_team_name/bio/edu/exp, story_proj_title/desc/meta,
          story_references, story_billing_rates


## Step 5 — Build Modified IDML & Export PDF

Run these two cells after all generation cells have completed successfully.


In [31]:
# ── Build Modified IDML ───────────────────────────────────────────────────────
# Maps every generated section to its IDML story file(s).

_today = date.today().strftime("%B %d, %Y")

STORY_MAP = {
    # ── Cover page fields ─────────────────────────────────────────────────────
    'Stories/Story_u1c6.xml':  lambda x: rc(x, 'RFP Title', cover['rfp_title']),
    'Stories/Story_u1ad.xml':  lambda x: rc(x, 'May 30, 2025', _today),
    'Stories/Story_u1df.xml':  lambda x: rc(x, 'Entity / client', cover['client_name']),
    'Stories/Story_u26c.xml':  lambda x: rc(x, 'Month Day, Year', _today),
    'Stories/Story_u2279.xml': lambda x: rc(x, 'RFP Title  |  Collab Architecture',
                                             f'{cover["rfp_title"]}  |  Collab Architecture'),
    'Stories/Story_u28b.xml':  lambda x: rc(x, 'RFP Title  |  Collab Architecture',
                                             f'{cover["rfp_title"]}  |  Collab Architecture'),

    # ── Cover letter ──────────────────────────────────────────────────────────
    'Stories/Story_u221.xml':  lambda x: story_cover_letter(cover),

    # ── Firm profile ──────────────────────────────────────────────────────────
    'Stories/Story_u1e69.xml': lambda x: rc(x,
        'Designing Spaces That Bring Communities Together', firm['headline']),
    'Stories/Story_u2054.xml': lambda x: rc(x,
        'Designing Spaces that Bring Communities Together.', firm['headline'] + '.'),
    'Stories/Story_u1e50.xml': lambda x: wrap_story('u1e50',
        pb('WHO WE ARE', 'Body', 'Bold, Teal') + '\n' + pbs(firm['who_we_are'], 'Body')),
    'Stories/Story_u2021.xml': lambda x: wrap_story('u2021',
        pbs(firm['who_we_are'] + '\n\n' + firm.get('differentiator', ''), 'Body')),
    'Stories/Story_u203b.xml': lambda x: wrap_story('u203b',
        pbs(approach['intro'], 'Body')),

    # ── Team: Jordan Lockner ──────────────────────────────────────────────────
    'Stories/Story_u67f.xml':  lambda x: story_team_name('u67f', team['lockner']),
    'Stories/Story_u665.xml':  lambda x: story_team_bio('u665',  team['lockner']),
    'Stories/Story_u64c.xml':  lambda x: story_team_edu('u64c',  team['lockner']),
    'Stories/Story_u69e.xml':  lambda x: story_team_exp('u69e',  team['lockner']),

    # ── Team: Kala Bailor ─────────────────────────────────────────────────────
    'Stories/Story_u703.xml':  lambda x: story_team_name('u703', team['bailor']),
    'Stories/Story_u6ea.xml':  lambda x: story_team_bio('u6ea',  team['bailor']),
    'Stories/Story_u6d0.xml':  lambda x: story_team_edu('u6d0',  team['bailor']),
    'Stories/Story_u71f.xml':  lambda x: story_team_exp('u71f',  team['bailor']),

    # ── Team: Michael Aller ───────────────────────────────────────────────────
    'Stories/Story_u918.xml':  lambda x: story_team_name('u918', team['aller']),
    'Stories/Story_u8b3.xml':  lambda x: story_team_bio('u8b3',  team['aller']),
    'Stories/Story_u84e.xml':  lambda x: story_team_edu('u84e',  team['aller']),
    'Stories/Story_u986.xml':  lambda x: story_team_exp('u986',  team['aller']),

    # ── Project experience 1 ──────────────────────────────────────────────────
    'Stories/Story_u2ef.xml':  lambda x: story_proj_title('u2ef', experience['project1']),
    'Stories/Story_u309.xml':  lambda x: story_proj_desc('u309',  experience['project1']),
    'Stories/Story_u322.xml':  lambda x: story_proj_meta('u322',  experience['project1']),

    # ── Project experience 2 ──────────────────────────────────────────────────
    'Stories/Story_u3eb.xml':  lambda x: story_proj_title('u3eb', experience['project2']),
    'Stories/Story_u404.xml':  lambda x: story_proj_desc('u404',  experience['project2']),
    'Stories/Story_u41d.xml':  lambda x: story_proj_meta('u41d',  experience['project2']),

    # ── References ────────────────────────────────────────────────────────────
    'Stories/Story_u1552.xml': lambda x: story_references(approach['references']),

    # ── Billing rates ─────────────────────────────────────────────────────────
    'Stories/Story_u1d57.xml': lambda x: story_billing_rates(fee),
    'Stories/Story_u1d70.xml': lambda x: rc(
        x,
        'Printing Services, Materials, SuppliesCost + 15%Mileage$.070 / mile',
        '  |  '.join(
            f'{r.get("item","")}: {r.get("basis","")}'
            for r in fee.get('reimbursables', [])
        ) or 'Outside Materials/Supplies: Cost + 15%  |  Mileage: $0.70/mile',
    ),
}

print(f"Applying {len(STORY_MAP)} story updates...\n")
IDML_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

_applied, _failed = 0, 0
with zipfile.ZipFile(str(IDML_TEMPLATE), 'r') as src:
    with zipfile.ZipFile(str(IDML_OUTPUT), 'w', zipfile.ZIP_DEFLATED) as dst:
        for item in src.infolist():
            raw = src.read(item.filename)
            if item.filename in STORY_MAP:
                try:
                    new_xml = STORY_MAP[item.filename](raw.decode('utf-8'))
                    dst.writestr(item, new_xml.encode('utf-8'))
                    print(f"  ✓  {item.filename.split('/')[-1]}")
                    _applied += 1
                except Exception as _e:
                    print(f"  ✗  {item.filename.split('/')[-1]}: {_e}")
                    dst.writestr(item, raw)
                    _failed += 1
            else:
                dst.writestr(item, raw)

print(f"\n{'=' * 50}")
print(f"  Applied: {_applied} stories  |  Errors: {_failed}")
print(f"  IDML saved → {IDML_OUTPUT}")

# ── Quick content verification ─────────────────────────────────────────────────
def _peek(story_file, max_chars=100):
    with zipfile.ZipFile(str(IDML_OUTPUT), 'r') as z:
        xml = z.read(story_file).decode('utf-8')
    return ' '.join(re.findall(r'<Content>([^<]+)</Content>', xml))[:max_chars]

print("\nSpot-checks:")
for label, sid in [
    ("RFP Title",     "Stories/Story_u1c6.xml"),
    ("Cover letter",  "Stories/Story_u221.xml"),
    ("Jordan name",   "Stories/Story_u67f.xml"),
    ("Project 1",     "Stories/Story_u2ef.xml"),
    ("Project 2",     "Stories/Story_u3eb.xml"),
    ("Billing rates", "Stories/Story_u1d57.xml"),
]:
    text = _peek(sid)
    ok = "✓" if text.strip() else "✗"
    print(f"  {ok}  {label:15s}: {text[:85]}")


Applying 33 story updates...

  ✓  Story_u2279.xml
  ✓  Story_u2054.xml
  ✓  Story_u203b.xml
  ✓  Story_u2021.xml
  ✓  Story_u1e69.xml
  ✓  Story_u1e50.xml
  ✓  Story_u1d70.xml
  ✓  Story_u1d57.xml
  ✓  Story_u1552.xml
  ✓  Story_u986.xml
  ✓  Story_u918.xml
  ✓  Story_u8b3.xml
  ✓  Story_u84e.xml
  ✓  Story_u71f.xml
  ✓  Story_u703.xml
  ✓  Story_u6ea.xml
  ✓  Story_u6d0.xml
  ✓  Story_u69e.xml
  ✓  Story_u67f.xml
  ✓  Story_u665.xml
  ✓  Story_u64c.xml
  ✓  Story_u41d.xml
  ✓  Story_u404.xml
  ✓  Story_u3eb.xml
  ✓  Story_u322.xml
  ✓  Story_u309.xml
  ✓  Story_u2ef.xml
  ✓  Story_u28b.xml
  ✓  Story_u221.xml
  ✓  Story_u1df.xml
  ✓  Story_u1c6.xml
  ✓  Story_u1ad.xml
  ✓  Story_u26c.xml

  Applied: 33 stories  |  Errors: 0
  IDML saved → /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/rfp_filled_template_v2.idml

Spot-checks:
  ✓  RFP Title      : Cascade Campus Facility Renovation
  ✓  Cover letter   : City of Loveland Public Works Department / Fa

In [32]:
# ── Export All Generated Text to Markdown ─────────────────────────────────────
# Saves the full proposal as generated_proposal.md in the same folder as this
# notebook, and renders a live preview below.

from IPython.display import display, Markdown as _MD
from pathlib import Path

_save_path = Path(r"/Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture") / "generated_proposal.md"
_today_str = date.today().strftime("%B %d, %Y")

_sections = []

# Cover letter
_sections.append(f"""# {cover.get('rfp_title','RFP Response')}
**{cover.get('rfp_number','')}  |  Collab Architecture  |  {_today_str}**

---

## Cover Letter

{cover.get('client_name','')}  
{cover.get('client_dept','')}  
{cover.get('client_address1','')}  
{cover.get('client_address2','')}  

{cover.get('re_line','')}

{cover.get('salutation','')}

{cover.get('body','')}

Sincerely,

**Jordan W. Lockner, AIA, NCARB**  
Founding Principal | Collab Architecture  
jordan@collabarchitects.com | 970.215.9907""")

# Firm profile
_sections.append(f"""---

## Firm Profile

### {firm.get('headline','')}

{firm.get('who_we_are','')}

**Why Collab**

{firm.get('differentiator','')}""")

# Team bios
def _bio_block(key):
    m = team.get(key, {})
    exp = "\n".join(f"- {p}" for p in m.get('experience', []))
    return f"""### {m.get('name','')}
**{m.get('title','')}**

{m.get('bio','')}

*Education:* {m.get('education','')}  
*Registrations:* {m.get('registrations','')}

**Select Experience**

{exp}"""

_sections.append("---\n\n## Project Team\n\n" +
    "\n\n---\n\n".join(_bio_block(k) for k in ["lockner","bailor","aller"]))

# Project experience
def _proj_block(key):
    p = experience.get(key, {})
    return f"""### {p.get('title','')}
**{p.get('location','')}**  |  Lead: {p.get('team_lead','')}  
*Type:* {p.get('project_type','')}  
*Services:* {p.get('services','')}

{p.get('description','')}"""

_sections.append("---\n\n## Relevant Project Experience\n\n" +
    "\n\n---\n\n".join(_proj_block(k) for k in ["project1","project2"]))

# Approach + references
_ref_list = "\n\n".join(
    "**" + ref.split("\n")[0] + "**  \n" + (ref.split("\n")[1] if "\n" in ref else "")
    for ref in approach.get('references', [])
)
_sections.append(f"""---

## Project Understanding & Approach

{approach.get('intro','')}

**Key Challenge & Mitigation**

{approach.get('key_challenges','')}

**Schedule**

{approach.get('schedule_intro','')}

---

## References

{_ref_list}""")

# Fee schedule
_rates = "\n".join(f"| {r.get('role','')} | {r.get('rate','')} |" for r in fee.get('rates',[]))
_reimb = "\n".join(f"| {r.get('item','')} | {r.get('basis','')} |" for r in fee.get('reimbursables',[]))
_sections.append(f"""---

## Fee Schedule

{fee.get('narrative','')}

### Standard Hourly Rates

| Role | Rate |
|------|------|
{_rates}

### Reimbursable Expenses

| Item | Basis |
|------|-------|
{_reimb}""")

_full_md = "\n\n".join(_sections)
_save_path.write_text(_full_md, encoding="utf-8")
print(f"✓  Saved → {_save_path}")
print(f"   {len(_full_md):,} characters")
print()
display(_MD(_full_md))


✓  Saved → /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/generated_proposal.md
   22,087 characters



# Cascade Campus Facility Renovation
**Bid No. 2025-075  |  Collab Architecture  |  June 08, 2026**

---

## Cover Letter

City of Loveland  
Public Works Department / Facilities Division  
105 West 5th Street  
Loveland, CO 80537  

Re: Bid No. 2025-075 — Architectural and Engineering Services for Cascade Campus Facility Renovation

To The City of Loveland Selection Committee & Stakeholders,

After reviewing the City of Loveland's Request for Qualifications, attending the mandatory pre-submittal meeting at 1515 Cascade Avenue, and evaluating the project's objectives, we recognize the significance of this renovation in supporting the long-term growth of the Loveland Utilities Department. Our team understands that this phase will modernize approximately 62,030 square feet across the first and second floors to accommodate the Department's 15-year staffing projections, encompassing interior finishes, space planning to City standards, collaborative teaming and conference spaces, MEP and building system upgrades, ADA compliance improvements, and exterior site enhancements such as parking, lighting, and EV charging. Equally important, we recognize that portions of the Cascade Campus will remain occupied by existing tenants and select City staff throughout design and construction, requiring a thoughtful, safety-driven phasing strategy developed in close collaboration with the City's project team and the selected general contractor.

Collab Architecture, together with our trusted engineering partners, brings directly relevant experience in municipal renovations, office space planning, programming for staffing projections, and the coordination of structural, mechanical, electrical, plumbing, and fire protection upgrades within active, occupied facilities. Our partnership with the City of Aurora on the 289,000-square-foot Municipal Center Space Planning and Remodel demonstrated our ability to assess multi-floor facilities, optimize layouts across numerous departments, and phase renovations around ongoing operations, while our Adams County Western Service Center 3rd Floor Programming project reflects our strength in ADA-driven modernization, high-use material selection, and stakeholder-led space planning. As a Northern Colorado firm with more than 30 completed projects in Loveland, we bring deep familiarity with the City's codes, permitting processes, construction and finish standards, and the design-bid-build delivery method anticipated for this work.

I, Jordan Lockner, will serve as the Principal Architect and primary firm contact, providing strategic oversight and alignment with the City's goals throughout design and construction. Project Architect Kala Bailor will serve as the Project Manager and primary day-to-day point of contact, leading programming verification, space planning, stakeholder coordination, and phasing efforts—drawing on her experience delivering municipal and public-safety facilities such as the Town of Timnath Police Services Building and Town Center. Michael Aller, with more than four decades of municipal and institutional design experience, will lead Quality Assurance and Quality Control, ensuring every deliverable meets City standards, ADA requirements, and constructability expectations before progressing to the next phase.

We are committed to delivering a functional, cost-effective, and phased renovation that supports the Loveland Utilities Department's evolving needs while maintaining safe, uninterrupted operations for existing tenants and staff. We appreciate your consideration and welcome the opportunity to present our team and approach in greater detail during the interview process.

Sincerely,

**Jordan W. Lockner, AIA, NCARB**  
Founding Principal | Collab Architecture  
jordan@collabarchitects.com | 970.215.9907

---

## Firm Profile

### Collaborative Design for Loveland's Growing Utilities Workforce

Collab Architecture is a Windsor-based architecture and interior design firm dedicated to bringing the power of collaborative design to create a stronger, better, and more sustainable community. Founded in 2020, our firm has rapidly established itself as a trusted partner for municipal clients throughout Northern Colorado and the Front Range, and we have been named one of the fastest-growing private companies in the region in both 2023 and 2024. Our holistic approach to every project begins with two fundamental questions: What design problem are we looking to solve, and how can we make the project the most successful for our client? It all starts with listening — "Stop. Collaborate and Listen" is our unofficial motto, and we push to live by those words on every engagement.

We provide full-service commercial architecture and interior design, with in-depth experience on projects ranging from interior tenant renovations and space planning to large ground-up civic structures. Our team brings directly relevant expertise to the Cascade Campus Facility Renovation, having led comparable efforts such as the City of Aurora Municipal Center Space Planning and Remodel — a 289,000 SF, five-story, twenty-department facility where we evaluated space utilization, addressed staffing growth, and reconfigured administrative and public-safety operations within an occupied building. Our work at the Adams County Western Service Center, the Town of Superior Downtown Civic Space, and the Department of Public Safety Administration & Training Facility further demonstrates our command of multi-floor interior renovations, MEP system upgrades, ADA compliance improvements, and modernization of finishes to support evolving programmatic needs.

We care deeply about the communities we serve and believe that creating spaces which enable society to thrive is at the heart of what we do. We work alongside our clients throughout the entire design process — from programming and concept through occupancy and closeout — to ensure that project goals are met and expectations are exceeded. This collaborative, listening-first approach has proven successful in mitigating preventable issues early in design, keeping projects on schedule and within budget, and delivering results that both our clients and our team are proud to stand behind.

**Why Collab**

Collab Architecture is uniquely positioned to serve the City of Loveland on the Cascade Campus Facility Renovation because our experience aligns precisely with the project's defining challenges — renovating roughly 62,030 SF across two floors of an occupied, multi-tenant building to accommodate the Loveland Utilities Department's 15-year staffing projections. We have completed over 30 projects in Loveland and bring a deep working knowledge of the City's codes, permitting processes, and construction and finish standards, allowing us to streamline coordination and proactively address challenges. Our team's hands-on translation of staffing projections into functional space plans at the City of Aurora Municipal Center, combined with our demonstrated ability to phase work while maintaining ongoing operations and services within active facilities, directly answers the City's implementation strategy of keeping existing staff areas and tenants in place throughout design and construction. With Project Manager Kala Bailor, AIA, LEED GA leading day-to-day coordination, QA/QC Manager Michael Aller bringing more than four decades of municipal facility experience, and trusted local engineering partners Larsen Structural Design and Integrated MEP — whose principals have delivered projects including the Larimer County Police and Courts Addition and City of Loveland Fire Stations 3 and 4 — we offer the City a cohesive, locally rooted team ready to deliver a practical, cost-conscious, and adaptable renovation through the design-bid-build delivery method.

---

## Project Team

### Jordan W. Lockner, AIA, NCARB
**Principal Architect**

Jordan believes that the root of all good architecture must stem from and support the community it serves, an approach well-suited to the City of Loveland's vision for the Cascade Campus renovation in support of the growing Loveland Utilities Department. His ability to listen, understand, and collaborate with clients while coordinating and managing the design process has produced successful outcomes for municipal facilities throughout Northern Colorado, including the 51,500 SF Town of Windsor Public Works Campus and the 289,000 SF City of Aurora Municipal Center Space Planning effort. As Principal Architect and your primary firm contact, Jordan will provide strategic oversight to ensure the renovation aligns with the Department's 15-year staffing projections and the City's space standards. His public-sector leadership has been recognized with the 2022 University of Colorado ENVD Young Designer Award and a 2023 Northern Colorado 40 Under 40 Honoree designation.

*Education:* University of Colorado, B.ENVD (Architecture focus)  
*Registrations:* Licensed Architect, NCARB

**Select Experience**

- City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO
- Town of Windsor, Public Works Campus - Windsor, CO (previous firm)
- Town of Superior, Downtown Civic Space - Superior, CO
- Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
- Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
- Department of Public Safety, Admin & Training Facility - Windsor, CO
- Town of Estes Park, Transit Facility - Estes Park, CO
- Weld County, Grounds Building Design - Greeley, CO

---

### Kala Bailor, AIA, LEED GA
**Project Manager**

Kala has always been a person interested in both the sciences and the arts, caring deeply about the details that ensure clients receive the best of both form and function—an asset for a phased renovation of 62,030 SF that must modernize interior finishes, MEP systems, and ADA compliance while keeping existing tenants and City staff in operation. Her attention to detail has proven successful in keeping projects on schedule and within budget across municipal structures such as the 29,000 SF Town of Timnath Police Services Building and the 15,250 SF Timnath Town Center. As Project Manager and your day-to-day point of contact, Kala will lead programming verification, stakeholder coordination with the Utilities Department, space planning to City standards, and the phasing plan that safeguards staff, public, and construction workers throughout the design-bid-build process. She has demonstrated excellent communication and coordination skills and consistently built repeat work with clients including the Town of Timnath.

*Education:* University of Colorado - Denver, Master of Architecture  
*Registrations:* Licensed Architect, LEED Green Associate

**Select Experience**

- Town of Timnath, Police Services Building - Timnath, CO (previous firm)
- Town of Timnath, Town Center - Timnath, CO (previous firm)
- Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
- Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
- City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO
- Town of Superior, Downtown Civic Space - Superior, CO
- Eaton Public Library, Renovation & Addition - Eaton, CO

---

### Michael Aller, AIA, LEED AP
**QA/QC Manager**

With over four decades of experience in municipal and higher education facility design, Mick brings the seasoned oversight needed to deliver a technically complex renovation involving MEP upgrades, ADA improvements, and modernized finishes across two occupied floors of the Cascade Campus. He is well-versed in coordinating large-scale technical projects and will lead a meticulous review process to ensure all documentation complies with the City of Loveland's construction and finish standards, ADA requirements, and permitting expectations under the design-bid-build delivery method. As QA/QC Manager, Mick is dedicated to delivering high-quality, tailored designs that meet client standards while minimizing conflicts and clarifications during construction. His work has earned prestigious recognition, including the AIA Colorado Citation Award and the F.W. Dodge Silver Hard Hat Award.

*Education:* University of Michigan, Master of Architecture  
*Registrations:* Licensed Architect, NCARB, LEED Accredited Professional

**Select Experience**

- City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO
- Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
- Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
- Town of Silverthorne, Recreation Center Expansion - Silverthorne, CO
- Town of Estes Park, Transit Facility - Estes Park, CO
- Adams County, Honnen Facility Conditions Assessment - Brighton, CO
- Department of Public Safety, Admin & Training Facility - Windsor, CO

---

## Relevant Project Experience

### CITY OF AURORA, MUNICIPAL CENTER SPACE PLANNING & REMODEL
**Aurora, CO**  |  Lead: Jordan Lockner, AIA, NCARB  
*Type:* Interior Renovation, Space Planning, Department Reconfiguration  
*Services:* Architecture, Interior Design, Programming, Construction Administration

Our team began its partnership with the City of Aurora through a comprehensive space planning effort and audit of the 289,000-square-foot Aurora Municipal Center (AMC). This project required evaluating space utilization across five stories and 20 municipal departments to address the facility's full capacity and prepare for future growth. By assessing operational needs, hybrid work opportunities, and potential department consolidations, we created a road-map to enhance efficiency, streamline workflows, and support scalability within the AMC—precisely the kind of programming and multi-floor space planning the City of Loveland envisions for the 62,030 SF Cascade Campus renovation supporting the Utilities Department's 15-year staffing projections.

As the scope evolved, we executed a series of targeted interior renovations within an occupied, fully operational municipal building, relocating Police Administration into the AMC and transforming a vacated first-floor suite into a dedicated, secure Internal Affairs Suite with private offices, interview rooms, and open office areas. This experience directly parallels the Cascade Campus requirement to maintain existing staff areas, tenants, and services throughout design and construction while phasing modernization of finishes, MEP systems, and collaborative workspaces.

The result was a flexible, phased framework that allowed the City to consolidate key administrative functions and right-size departments without disrupting daily operations—establishing repeat work and an ongoing, trusted relationship with the City of Aurora across multiple subsequent renovation packages including IT, customer service, and conference center improvements.

---

### ADAMS COUNTY, WESTERN SERVICE CENTER 3RD FLOOR PROGRAMMING & RENOVATION
**Westminster, CO**  |  Lead: Kala Bailor, AIA, LEED GA  
*Type:* Interior Renovation, ADA Compliance, Finish Modernization  
*Services:* Architecture, Interior Design, Programming, Construction Administration

Our team is collaborating with Adams County to renovate the Western Services Building, a high-use government facility, with a focus on enhancing universal accessibility and ADA compliance—a core requirement of the Cascade Campus scope, which calls for ADA improvements alongside modernization of finishes and systems. The project encompasses the transformation of restroom facilities across all floors into non-gender, family-friendly spaces that comply with current accessibility standards for government facilities, demonstrating our team's direct experience designing public buildings that meet federal and state ADA requirements.

The scope also included upgrades and a redesign of the front lobby, elevator lobbies, and main hallways to create a more welcoming environment, along with careful material selections evaluated for ease of maintenance, long-term durability, and aesthetics. This mirrors the Cascade Campus emphasis on updated interior finishes—carpeting, paint, casework, ceiling tiles, restroom fixtures, and window shades—using durable, maintainable materials appropriate to a public facility supporting daily operations.

By delivering ADA-compliant, modernized spaces within a fully occupied county facility, our team produced a renovation that improved both accessibility and visitor experience without major structural changes—a model directly applicable to the City's need to renovate Cascade Campus while existing tenants and City staff remain in place.

---

## Project Understanding & Approach

After reviewing the City of Loveland's RFQ for the Cascade Campus Facility Renovation, attending the mandatory pre-submittal meeting, and evaluating the project's objectives, we recognize the strategic importance of transforming the newly acquired 1515 Cascade Avenue facility into a modern, functional home for the Loveland Utilities Department. This renovation of approximately 62,030 SF across the first and second floors represents a critical investment in supporting the Department's projected 15-year staffing growth, building upon the programming efforts the City initiated in 2023. Our team is prepared to deliver a thoughtful renovation that addresses the full scope outlined in the RFQ, including interior finish updates, space planning and furniture layouts per City standards, collaborative teaming and conference spaces with audio/visual improvements, break areas and outdoor amenities, comprehensive MEP and network connectivity upgrades, ADA compliance improvements, and exterior site enhancements such as parking layout, lighting, EV charging, and security improvements. Through strategic programming, detailed space planning, and a collaborative approach with both the Facilities Division and Utilities Department stakeholders, we will create an adaptable solution that supports the Department's long-term operational needs.

We understand that the defining condition of this project is the requirement to maintain existing operations within an occupied facility—the Cascade Campus will house existing outside tenants and select City staff throughout both design and construction. This demands a carefully sequenced phasing strategy, developed collaboratively with the City's project team and the pre-qualified general contractor, that assures the safety of City employees, the public, and construction workers while keeping services uninterrupted. Our approach also accounts for the Design-Bid-Build delivery method, the one percent (1%) for the Arts coordination with the City's Cultural Services Department, delegated-design fire protection during construction, and the City's construction and finish standards. With extensive experience in municipal renovations, administrative space reconfigurations, programming for staffing projections, and technical facility upgrades, our team is well-equipped to assess existing conditions, implement targeted design solutions, and deliver a high-performing, cost-controlled project from conceptual design through closeout.

**Key Challenge & Mitigation**

The single biggest challenge on this project is executing a comprehensive two-floor renovation within a facility that remains occupied by existing tenants and City staff throughout design and construction, where any disruption to safety or service is unacceptable. Our mitigation strategy centers on developing a detailed, collaboratively vetted phasing plan early in the project—working directly with the Department, existing tenants, the City's project management staff, and the selected general contractor to establish safe construction zones, maintain occupant access, and sequence work to minimize operational impact. We will reinforce this plan with structured communication tools, regular site coordination, and clear milestone checkpoints to ensure the phasing approach remains responsive as the design and construction progress.

**Schedule**

Our proposed schedule aligns with the City's anticipated timeline of design commencing in Quarter 1 2026 with construction slated to begin in 2027, providing a structured, logic-driven progression from programming through project closeout. We will work closely with the City's Facilities Division and Utilities Department stakeholders to refine milestones, establish phasing expectations, and ensure an efficient advancement through each design and construction phase.

---

## References

**Brian Rowe, Deputy Director of Public Works, Town of Windsor**  
970.674.5400  |  browe@windsorgov.com

**Elly Watson, Business Services Manager, City of Aurora**  
303.739.7109  |  elwatson@auroragov.org

**Kyle Burg, Project Manager, Facilities & Fleet Management, Adams County**  
720.523.6062  |  KBurg@adcogov.org

---

## Fee Schedule

Our fee is structured on a unit-price basis with a not-to-exceed total aligned to the City of Loveland's purchasing requirements and the Cascade Campus Facility Renovation scope, from conceptual programming through project closeout. Each phase—conceptual design and programming, schematic design, design development, construction documents, bidding and procurement, and construction administration—carries its own budget cap, allowing the City to track financial commitments against deliverables at every milestone. We will provide monthly earned-value reporting that ties invoiced amounts to percentage of work completed, ensuring full transparency and proactive cost control throughout the design-bid-build process.

### Standard Hourly Rates

| Role | Rate |
|------|------|
| Principal Architect / Engineer | $225 / hour |
| Project Architect / Engineer | $205 / hour |
| Project Manager / Engineer | $185 / hour |
| QA/QC Review | $185 / hour |
| CAD Technician | $115 / hour |
| Interior Designer | $95 / hour |
| Administrative | $75 / hour |

### Reimbursable Expenses

| Item | Basis |
|------|-------|
| Outside Materials / Services / Supplies | Cost + 15% |
| Mileage | $0.70 / mile |

In [33]:
# ── Open Filled IDML for Manual PDF Export ────────────────────────────────────
# Run this cell to open the IDML in InDesign, then export to PDF manually.

print("=" * 58)
print("  MANUAL EXPORT STEPS")
print("=" * 58)
print(f"\n  File: {IDML_OUTPUT}\n")
print("  1. InDesign will open with the filled template")
print("  2. File  →  Export")
print("  3. Format: Adobe PDF (Print)")
print("  4. Filename: example_rfq_to_rfp_proposal.pdf")
print("  5. Click Export → keep defaults → Export\n")

_open = subprocess.run(['open', str(IDML_OUTPUT)], capture_output=True, text=True)
if _open.returncode == 0:
    print("✓  File opened — follow the steps above in InDesign.")
else:
    print(f"Could not open automatically. Open manually:\n  {IDML_OUTPUT}")


  MANUAL EXPORT STEPS

  File: /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/rfp_filled_template_v2.idml

  1. InDesign will open with the filled template
  2. File  →  Export
  3. Format: Adobe PDF (Print)
  4. Filename: example_rfq_to_rfp_proposal.pdf
  5. Click Export → keep defaults → Export

Could not open automatically. Open manually:
  /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/rfp_filled_template_v2.idml


In [34]:
# ── Export PDF via Adobe InDesign ─────────────────────────────────────────────
# POSIX file must be declared OUTSIDE the tell block.
# Uses 'export format PDF type' without 'showing options' for compatibility.

_idml_str = str(IDML_OUTPUT)
_pdf_str  = str(PDF_OUTPUT)


def _try_indesign(app_name):
    script = f"""set theIDML to POSIX file \"{_idml_str}\"
tell application \"{app_name}\"
    activate
    set myDoc to open theIDML
    tell myDoc
        export format PDF type to \"{_pdf_str}\"
    end tell
    close myDoc saving no
end tell"""
    return subprocess.run(['osascript', '-e', script], capture_output=True, text=True, timeout=300)


print(f"Exporting to PDF...")
print(f"  Source IDML : {_idml_str}")
print(f"  Output PDF  : {_pdf_str}\n")

_success = False
for _app in ["Adobe InDesign 2025", "Adobe InDesign 2024", "Adobe InDesign 2023"]:
    print(f"  Trying {_app}...", end=" ", flush=True)
    _res = _try_indesign(_app)
    if _res.returncode == 0 and PDF_OUTPUT.exists():
        print(f"✓  ({PDF_OUTPUT.stat().st_size // 1024} KB)")
        subprocess.run(['open', _pdf_str])
        print(f"\n✅  PDF saved → {PDF_OUTPUT}")
        _success = True
        break
    else:
        _err = (_res.stderr or _res.stdout or "").strip()[:120]
        print(f"✗  {_err}" if _err else "✗  not found")

if not _success:
    print("\n⚠️  InDesign not found. Run the cell above to open the IDML manually.")


Exporting to PDF...
  Source IDML : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/rfp_filled_template_v2.idml
  Output PDF  : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/example_rfq_to_rfp_proposal.pdf

  Trying Adobe InDesign 2025... ✗  251:257: syntax error: Expected end of line but found identifier. (-2741)
  Trying Adobe InDesign 2024... ✗  251:257: syntax error: Expected end of line but found identifier. (-2741)
  Trying Adobe InDesign 2023... ✗  251:257: syntax error: Expected end of line but found identifier. (-2741)

⚠️  InDesign not found. Run the cell above to open the IDML manually.
